# CNN Feature Extraction + SVM Classification Comparison

**Muc tieu:** So sanh hieu suat phan loai giua CNN thuan (Softmax) va CNN+SVM cho bai toan nhan dien benh tren la sau rieng.

**Cac model:** ResNet18, ResNet34, ResNet50, MobileNetV3-Large, EfficientNet-B0

---

## Huong dan tren Kaggle (QUAN TRONG)

**BUOC 1:** Tren menu, vao **"Runtime" > "Restart runtime"** (Ctrl+M.)

**BUOC 2:** O dau Cell 1, dien **duong dan TUYET DOI** cua ban:
```python
MANUAL_PROJECT_ROOT = "/kaggle/input/datasets/tranmanh1312/dataset-source-code-latest/ai-durian-disease-detection-2"
MANUAL_DATA_ROOT    = "/kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf"
```

**BUOC 3:** Chay **Cell 1**, neu OK chay tiep Cell 2 va cac cell con lai

In [ ]:
# CELL 1: cai dat thu vien + duong dan
# ============================================================
#  DIEN 2 DONG DUOI DAY VOI DUONG DAN TUYET DOI CUA BAN
# ============================================================
MANUAL_PROJECT_ROOT = "/kaggle/input/datasets/tranmanh1312/dataset-source-code-latest/ai-durian-disease-detection-2"   # <-- SUA DONG NAY
MANUAL_DATA_ROOT    = "/kaggle/input/datasets/tranmanh1312/durian-processed-data/data-leaf"                         # <-- SUA DONG NAY
# ============================================================

import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle")
print(f"Running on Kaggle: {IS_KAGGLE}")

# ---- Tu dong tim project root (quet bat ky do sau nao) ----
def find_project_root():
    if IS_KAGGLE:
        input_dir = Path("/kaggle/input")
        if input_dir.exists():
            for src_dir in input_dir.rglob("src"):
                proj = src_dir.parent
                if (proj / "configs").is_dir():
                    return proj
            for src_dir in input_dir.rglob("src"):
                return src_dir.parent
    return None

# ---- Tu dong tim data root ----
def find_data_root():
    if IS_KAGGLE:
        input_dir = Path("/kaggle/input")
        if input_dir.exists():
            for train_dir in input_dir.rglob("train"):
                root = train_dir.parent
                if (root / "val").is_dir() and (root / "test").is_dir():
                    return root
            for train_dir in input_dir.rglob("train"):
                return train_dir.parent
    return None

# ---- Xac dinh PROJECT_ROOT ----
if MANUAL_PROJECT_ROOT:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)
    print(f"[MANUAL] Project root: {PROJECT_ROOT}")
else:
    PROJECT_ROOT = find_project_root()
    print(f"[AUTO] Project root: {PROJECT_ROOT}")

if PROJECT_ROOT is None or not (PROJECT_ROOT / "src").is_dir():
    print("\n[AUTO-DETECT FAILED] Vui long dien MANUAL_PROJECT_ROOT o dau Cell 1!")
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        for d1 in sorted(input_dir.iterdir()):
            if d1.is_dir():
                print(f"  - {d1}")
                for d2 in sorted(d1.iterdir()):
                    if d2.is_dir():
                        print(f"    - {d2}")
else:
    print(f"  src/ OK: {(PROJECT_ROOT / 'src').exists()}")
    print(f"  configs/ OK: {(PROJECT_ROOT / 'configs').exists()}")

# ---- Them env var + sys.path (CHO CONFIG.PY) ----
if PROJECT_ROOT and (PROJECT_ROOT / "src").is_dir():
    os.environ["AI_PROJECT_ROOT"] = str(PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"[OK] AI_PROJECT_ROOT = {PROJECT_ROOT}")
else:
    print("[ERROR] Vui long dat MANUAL_PROJECT_ROOT!")

# ---- Xac dinh DATA_ROOT ----
if MANUAL_DATA_ROOT:
    DATA_ROOT = Path(MANUAL_DATA_ROOT)
    print(f"[MANUAL] Data root: {DATA_ROOT}")
else:
    DATA_ROOT = find_data_root()
    if DATA_ROOT is None:
        print("[AUTO] Khong tu dong tim thay data root!")
        DATA_ROOT = PROJECT_ROOT / "data/processed" if PROJECT_ROOT else None
    else:
        print(f"[AUTO] Data root: {DATA_ROOT}")

TRAIN_DIR = DATA_ROOT / "train" if DATA_ROOT else None
VAL_DIR   = DATA_ROOT / "val"   if DATA_ROOT else None
TEST_DIR  = DATA_ROOT / "test"  if DATA_ROOT else None

print(f"  Train: {TRAIN_DIR} ({'OK' if TRAIN_DIR and TRAIN_DIR.exists() else 'MISSING'})")
print(f"  Val:   {VAL_DIR} ({'OK' if VAL_DIR and VAL_DIR.exists() else 'MISSING'})")
print(f"  Test:  {TEST_DIR} ({'OK' if TEST_DIR and TEST_DIR.exists() else 'MISSING'})")

# ---- Debug: kiem tra noi dung src/ tren Kaggle ----
if IS_KAGGLE and PROJECT_ROOT:
    src_dir = PROJECT_ROOT / "src"
    svm_dir = src_dir / "svm"
    print(f"\n[DEBUG] src/ exists: {src_dir.exists()}")
    if src_dir.exists():
        print(f"[DEBUG] Contents of src/:")
        for item in sorted(src_dir.iterdir()):
            print(f"  - {item.name}{'/ (dir)' if item.is_dir() else ''}")
    print(f"[DEBUG] src/svm/ exists: {svm_dir.exists()}")

In [ ]:
# CELL 2: Import thu vien

import logging
import time
import random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_auc_score, classification_report
)

# ---- Try to import src.svm; fall back to inline constants if missing ----
try:
    from src.svm.config import (
        DISEASE_CLASSES, DISEASE_CLASSES_VI,
        MODEL_REGISTRY, FEATURE_CACHE_DIR, RESULTS_DIR, RESULTS_CM_DIR,
        SVM_MODELS_DIR, RANDOM_STATE, BATCH_SIZE, OUTPUT_ROOT
    )
    from src.svm.feature_extractor import CNNFeatureExtractor
    from src.svm.svm_pipeline import SVMClassifier
    from src.svm.evaluate import plot_confusion_matrix, plot_comparison_chart, print_results_table, save_results_csv
    print(f"[OK] Loaded from src.svm/")

except (ModuleNotFoundError, ImportError) as e:
    print(f"[WARN] src.svm/ not found in dataset: {e}")
    print(f"[FALLBACK] Using inline constants — make sure src/svm/ is included in your Kaggle dataset!")

    # ---- Disease classes ----
    DISEASE_CLASSES = [
        "ALGAL_LEAF_SPOT", "ALLOCARIDARA_ATTACK", "HEALTHY_LEAF",
        "LEAF_BLIGHT", "PHOMOPSIS_LEAF_SPOT",
    ]
    DISEASE_CLASSES_VI = {
        "ALGAL_LEAF_SPOT": "Ben dot tao",
        "ALLOCARIDARA_ATTACK": "Bo tri tan cong",
        "HEALTHY_LEAF": "La khoe manh",
        "LEAF_BLIGHT": "Benh chay la",
        "PHOMOPSIS_LEAF_SPOT": "Benh dom la Phomopsis",
    }

    # ---- Output root (same logic as config.py) ----
    if os.path.exists("/kaggle"):
        OUTPUT_ROOT = Path("/kaggle/working")
        try:
            (OUTPUT_ROOT / ".write_test").touch()
            (OUTPUT_ROOT / ".write_test").unlink()
        except OSError:
            OUTPUT_ROOT = Path("/tmp/kaggle_output")
            OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    else:
        OUTPUT_ROOT = PROJECT_ROOT or Path(".")

    FEATURE_CACHE_DIR = OUTPUT_ROOT / "features"
    RESULTS_DIR       = OUTPUT_ROOT / "results" / "svm_comparison"
    RESULTS_CM_DIR    = RESULTS_DIR / "confusion_matrices"
    SVM_MODELS_DIR    = OUTPUT_ROOT / "models" / "svm_models"
    for _d in [FEATURE_CACHE_DIR, RESULTS_DIR, RESULTS_CM_DIR, SVM_MODELS_DIR]:
        Path(_d).mkdir(parents=True, exist_ok=True)

    RANDOM_STATE = 42
    BATCH_SIZE   = 32

    # ---- Model registry (must match src/svm/config.py) ----
    from dataclasses import dataclass
    @dataclass
    class ModelInfo:
        model_name: str
        display_name: str
        checkpoint_path: str
        config_path: str
        feature_dim: int
        image_size: int
        classifier_attr: str
        family: str
        def checkpoint_exists(self): return Path(self.checkpoint_path).exists()

    MODEL_REGISTRY = {
        "resnet18": ModelInfo("resnet18","ResNet18",
            str(PROJECT_ROOT/"models/checkpoints/resnet18/best_model.pth"),
            str(PROJECT_ROOT/"configs/config_resnet18.yaml"),
            512, 224, "fc", "resnet"),
        "resnet34": ModelInfo("resnet34","ResNet34",
            str(PROJECT_ROOT/"models/checkpoints/resnet34/best_model.pth"),
            str(PROJECT_ROOT/"configs/config_resnet34.yaml"),
            512, 224, "fc", "resnet"),
        "resnet50": ModelInfo("resnet50","ResNet50",
            str(PROJECT_ROOT/"models/checkpoints/resnet50/best_model.pth"),
            str(PROJECT_ROOT/"configs/config_resnet50.yaml"),
            2048, 256, "fc", "resnet"),
        "mobilenetv3_large": ModelInfo("mobilenetv3_large","MobileNetV3-Large",
            str(PROJECT_ROOT/"models/checkpoints/mobilenetv3_large/best_model.pth"),
            str(PROJECT_ROOT/"configs/config_mobilenetv3.yaml"),
            960, 224, "classifier", "mobilenet"),
        "efficientnet_b0": ModelInfo("efficientnet_b0","EfficientNet-B0",
            str(PROJECT_ROOT/"models/checkpoints/efficientnet_b0/best_model.pth"),
            str(PROJECT_ROOT/"configs/config_efficientnet_b0.yaml"),
            1280, 224, "classifier", "efficientnet"),
    }

    # ---- Stub functions when src/svm/ is missing ----
    def get_model_info(name):
        return MODEL_REGISTRY[name]

    class CNNFeatureExtractor:
        raise RuntimeError(
            "src.svm/ is missing from the dataset. "
            "Please update your Kaggle dataset to include src/svm/ "
            "or re-upload the latest version of your source code."
        )

    class SVMClassifier:
        raise RuntimeError(
            "src.svm/ is missing from the dataset. "
            "Please update your Kaggle dataset to include src/svm/ "
            "or re-upload the latest version of your source code."
        )

    def plot_confusion_matrix(*args, **kwargs): pass
    def plot_comparison_chart(*args, **kwargs): return plt.figure()
    def print_results_table(*args, **kwargs): pass
    def save_results_csv(*args, **kwargs): pass

print(f"OUTPUT_ROOT (writable): {OUTPUT_ROOT}")
print(f"FEATURE_CACHE_DIR: {FEATURE_CACHE_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
print(f"Random seed: {RANDOM_STATE}")

---

## Phan 1: Trich xuat Features

In [ ]:
# CELL 3: Kiem tra checkpoint

MODEL_NAMES = ["resnet50", "efficientnet_b0", "mobilenetv3_large", "resnet34", "resnet18"]

available_models = []
for name in MODEL_NAMES:
    info = MODEL_REGISTRY[name]
    exists = info.checkpoint_exists()
    print(f"  {info.display_name}: {'OK' if exists else 'MISSING'} - {info.checkpoint_path}")
    if exists:
        available_models.append(name)

print(f"\nAvailable models: {available_models}")

if not available_models:
    print("\n[!] Khong co checkpoint nao!")

In [ ]:
# CELL 4: Trich xuat features

extracted_features = {}

for model_name in tqdm(available_models, desc="Extracting features"):
    print(f"\n{'='*50}")
    print(f"Model: {MODEL_REGISTRY[model_name].display_name} | dim={MODEL_REGISTRY[model_name].feature_dim}")
    print(f"{'='*50}")
    
    extractor = CNNFeatureExtractor(model_name, device=device)
    
    X_train, y_train = extractor.extract_from_dataset(str(TRAIN_DIR), cache_tag="train")
    X_val,   y_val   = extractor.extract_from_dataset(str(VAL_DIR),   cache_tag="val")
    X_test,  y_test  = extractor.extract_from_dataset(str(TEST_DIR),  cache_tag="test")
    
    print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"  Label dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    
    extracted_features[model_name] = (X_train, y_train, X_val, y_val, X_test, y_test)

print("\nAll features extracted successfully!")

---

## Phan 2: Danh gia CNN-Softmax

In [ ]:
# CELL 5: Danh gia CNN-Softmax

def evaluate_cnn_softmax(model_name, X_test, y_test, class_names, batch_size=32):
    from src.models.model_factory import build_model
    from src.models.utils import load_checkpoint
    from src.utils.config import load_config

    info = MODEL_REGISTRY[model_name]
    
    try:
        config_raw = load_config(info.config_path).raw
    except Exception as e:
        print(f"  Warning: {e}")
        return None
    
    model = build_model(config_raw).to(device)
    if info.checkpoint_exists():
        load_checkpoint(model, info.checkpoint_path, device)
    model.eval()
    
    test_tensor = torch.from_numpy(X_test).float()
    test_labels = torch.from_numpy(y_test).long()
    loader = DataLoader(TensorDataset(test_tensor, test_labels), batch_size=batch_size, shuffle=False)
    
    all_preds, all_probs = [], []
    t0 = time.time()
    
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
    
    elapsed_ms = (time.time() - t0) / len(y_test) * 1000
    
    acc = accuracy_score(y_test, all_preds)
    prec, rec, f1_m, _ = precision_recall_fscore_support(y_test, all_preds, average="macro", zero_division=0)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, all_preds, average="weighted", zero_division=0)
    cm = confusion_matrix(y_test, all_preds)
    
    try:
        auc = roc_auc_score(y_test, np.array(all_probs), multi_class="ovr", average="macro",
                           labels=list(range(len(class_names))))
    except Exception:
        auc = None
    
    return {
        "model_name": info.display_name,
        "mode": "cnn",
        "accuracy": float(acc),
        "f1_macro": float(f1_m),
        "f1_weighted": float(f1_w),
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "roc_auc": float(auc) if auc else None,
        "inference_ms": float(elapsed_ms),
        "confusion_matrix": cm,
        "labels": y_test.tolist(),
        "predictions": all_preds,
        "probabilities": all_probs,
    }

cnn_results = []
for model_name in tqdm(available_models, desc="CNN-Softmax"):
    print(f"\n>>> {MODEL_REGISTRY[model_name].display_name} CNN-Softmax")
    X_train, y_train, X_val, y_val, X_test, y_test = extracted_features[model_name]
    result = evaluate_cnn_softmax(model_name, X_test, y_test, DISEASE_CLASSES)
    if result:
        print(f"  Acc={result['accuracy']:.4f} | F1={result['f1_macro']:.4f} | AUC={result['roc_auc'] or 'N/A'}")
        cnn_results.append(result)

---

## Phan 3: Train SVM

In [ ]:
# CELL 6: Train SVM

svm_results = []

for model_name in tqdm(available_models, desc="Training SVM"):
    print(f"\n{'='*50}")
    print(f"SVM: {MODEL_REGISTRY[model_name].display_name}")
    print(f"{'='*50}")
    
    X_train, y_train, X_val, y_val, X_test, y_test = extracted_features[model_name]
    
    X_tv = np.vstack([X_train, X_val])
    y_tv = np.concatenate([y_train, y_val])
    
    print(f"  Train+Val: {X_tv.shape} | Test: {X_test.shape}")
    
    clf = SVMClassifier(model_tag=f"{model_name}_svm", verbose=False)
    clf.fit(X_tv, y_tv)
    
    print(f"  Best params: {clf.best_params} | CV F1: {clf.best_cv_score:.4f}")
    clf.save()
    
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    t0 = time.time()
    _ = clf.predict(X_test)
    elapsed_ms = (time.time() - t0) / len(y_test) * 1000
    
    try:
        auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro",
                           labels=list(range(len(DISEASE_CLASSES))))
    except Exception:
        auc = None
    
    result = {
        "model_name": f"{MODEL_REGISTRY[model_name].display_name}+SVM",
        "mode": "svm",
        "accuracy": float(acc),
        "f1_macro": float(f1_m),
        "f1_weighted": float(f1_w),
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "roc_auc": float(auc) if auc else None,
        "inference_ms": float(elapsed_ms),
        "confusion_matrix": cm,
        "labels": y_test.tolist(),
        "predictions": y_pred.tolist(),
        "probabilities": y_proba.tolist(),
        "best_params": clf.best_params,
        "cv_f1_macro": float(clf.best_cv_score),
    }
    
    print(f"  Test: Acc={acc:.4f} | F1={f1_m:.4f} | AUC={auc or 'N/A'}")
    svm_results.append(result)

---

## Phan 4: Tong hop ket qua

In [ ]:
# CELL 7: Tong hop ket qua

all_results = cnn_results + svm_results

print_results_table(all_results)

csv_path = str(RESULTS_DIR / "metrics_comparison.csv")
save_results_csv(all_results, save_path=csv_path)
print(f"\nCSV saved: {csv_path}")

rows = []
for r in all_results:
    rows.append({
        "Model": r["model_name"],
        "Mode": r["mode"],
        "Accuracy": f"{r['accuracy']:.4f}",
        "F1-Macro": f"{r['f1_macro']:.4f}",
        "F1-Weighted": f"{r['f1_weighted']:.4f}",
        "Precision": f"{r['precision_macro']:.4f}",
        "Recall": f"{r['recall_macro']:.4f}",
        "ROC-AUC": f"{r['roc_auc']:.4f}" if r["roc_auc"] else "N/A",
        "Infer(ms)": f"{r['inference_ms']:.3f}",
    })

display(pd.DataFrame(rows))

---

## Phan 5: Confusion Matrix

In [ ]:
# CELL 8: Confusion matrix

for result in cnn_results:
    short = result["model_name"].replace(" ", "_")
    plot_confusion_matrix(result["confusion_matrix"],
        class_names=DISEASE_CLASSES,
        title=f"{result['model_name']} - CNN",
        save_path=str(RESULTS_CM_DIR / f"{short}_cnn_cm.png"))

for result in svm_results:
    short = result["model_name"].replace("+", "_").replace(" ", "_")
    plot_confusion_matrix(result["confusion_matrix"],
        class_names=DISEASE_CLASSES,
        title=f"{result['model_name']} - SVM",
        save_path=str(RESULTS_CM_DIR / f"{short}_svm_cm.png"))

---

## Phan 6: Bieu do so sanh

In [ ]:
# CELL 9: Bieu do so sanh

fig = plot_comparison_chart(all_results, save_path=str(RESULTS_DIR / "comparison_chart.png"))
plt.show(fig)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(available_models))

ax = axes[0]
ax.bar(x - 0.2, [r["f1_macro"] for r in cnn_results], 0.4, label="CNN", color="#4C72B0")
ax.bar(x + 0.2, [r["f1_macro"] for r in svm_results], 0.4, label="CNN+SVM", color="#DD8452")
ax.set_ylabel("F1-Macro"); ax.set_title("F1-Macro", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([MODEL_REGISTRY[n].display_name for n in available_models], rotation=20, ha="right")
ax.legend(); ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3)

ax = axes[1]
ax.bar(x - 0.2, [r["inference_ms"] for r in cnn_results], 0.4, label="CNN", color="#4C72B0")
ax.bar(x + 0.2, [r["inference_ms"] for r in svm_results], 0.4, label="CNN+SVM", color="#DD8452")
ax.set_ylabel("ms / anh"); ax.set_title("Inference Time", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([MODEL_REGISTRY[n].display_name for n in available_models], rotation=20, ha="right")
ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "detailed_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

---

## Phan 7: Classification Report

In [ ]:
# CELL 10: Classification Report

for result in svm_results:
    print(f"\n{'='*60}")
    print(f"{result['model_name']} | CV F1: {result.get('cv_f1_macro', 'N/A'):.4f}")
    print(f"Best params: {result.get('best_params', {})}")
    print(f"{'='*60}")
    report = classification_report(result["labels"], result["predictions"],
        target_names=DISEASE_CLASSES, output_dict=True, zero_division=0)
    display(pd.DataFrame(report).transpose().round(4))

---

## Ket luan

1. **CNN+SVM vs CNN**: So sanh cot F1-macro de xem SVM co cai thien khong
2. **Feature quality**: ResNet50 (2048-d) thuong tot hon ResNet18 (512-d)
3. **Inference time**: SVM nhanh hon Softmax tren CPU
4. **De tai tiep**: PCA, XGBoost, different SVM kernels